In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
import pickle
import logging

from nichejepa.datasets import make_cell_dataset, prepare_dataset
from nichejepa.masks.block_masking import BlockMaskCollator
from nichejepa.utils.config import create_params_from_YAML_wandb_config

In [11]:
world_size = 1
rank = 0

In [12]:
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)
params = create_params_from_YAML_wandb_config('../../../nichejepa/configs/template.yaml',
                                              logger)
params["data"]["batch_size"] = 2

INFO:root:Loaded parameters from YAML file.


In [13]:
params

{'data': {'dataset_name': 'mmb0-1b_smb1-1b_1p',
  'raw_data_folder_path': '/lustre/scratch126/cellgen/team361/DATASETS/silver/mmb0-1b_smb1-1b_1p',
  'token_dict_folder_path': '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-graph-tokenizer/mmb0-1b_smb1-1b_1p/token_dictionary.pkl',
  'tokenized_data_folder_path': '/lustre/scratch126/cellgen/team361/DATASETS/gold/cell-graph-tokenizer/mmb0-1b_smb1-1b_1p/mmb0-1b_smb1-1b_1p_None_shifted_log_knn_10_1793.dataset',
  'tokenizer_type': 'cell_graph',
  'random_state': 0,
  'sample_size': 500,
  'sample_subset': False,
  'split': 0.2,
  'stratify': False,
  'seq_len_cell': 64,
  'seq_len_neighborhood': 640,
  'n_segments': 11,
  'sampling_strategy': None,
  'batch_size': 2,
  'num_workers': 6,
  'pin_memory': True},
 'meta': {'add_cls': True,
  'gt_type': 'counts',
  'enc_depth': 3,
  'enc_emb_dim': 768,
  'pred_depth': 1,
  'pred_emb_dim': 768,
  'special_tokens': ['batch'],
  'pos_learnable': False,
  'seg_learnable': False,
  'use_b

In [14]:
train_dataset, _ = prepare_dataset(params)
train_dataset

Dataset({
    features: ['cell_degrees', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value_token', 'gene_panel_value_token', 'assay_value_token', 'species_value_token', 'tissue_value_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens', 'cls_tokens'],
    num_rows: 69457
})

In [19]:
# Load token dict and get token dict-specfic params
with open(params["data"]["token_dict_folder_path"], 'rb') as file:
    token_dict = pickle.load(file)
vocab_size = len(token_dict)
n_special_values = sum(1 for key in token_dict if "spv" in key)
max_special_tokens = sum(1 for key in token_dict if "cls" in key) + sum(
    1 for key in token_dict if "spt" in key)

In [26]:
# Define tokenizer-specific params
if params["data"]["tokenizer_type"] == 'cell_neighborhood':
    if params["meta"]["add_cls"]:
        special_tokens = ['cls_cell', 'cls_neighborhood'] +  params['meta']['special_tokens']            
elif params["data"]["tokenizer_type"] == 'cell_graph':
    if params["meta"]["add_cls"]:
        special_tokens = [
            f'cls_{i}' for i in range(params["data"]["n_segments"])] +  params['meta']['special_tokens']

max_cls_tokens = sum('cls' in token for token in special_tokens)

# Get token sequence length and number of special tokens
n_special_tokens = len(special_tokens)
seq_len = params["data"]["seq_len_cell"] + params["data"]["seq_len_neighborhood"] + n_special_tokens

In [28]:
mask_collator = BlockMaskCollator(
    n_targets=params["mask"]["n_targets"],
    n_contexts=params["mask"]["n_contexts"],
    n_segments=params["data"]["n_segments"],
    seq_len_cell=params["data"]["seq_len_cell"],
    seq_len_neighborhood=params["data"]["seq_len_neighborhood"],
    max_special_tokens=max_special_tokens,
    n_special_tokens=n_special_tokens,
    max_cls_tokens=max_cls_tokens,
    per_block_mask_ratio=params["mask"]["per_block_mask_ratio"],
    controlled_attention_pattern=params["mask"]["controlled_attention_pattern"],
    controlled_attention_type=params["mask"]["controlled_attention_type"],
    restrict_special_attention=params["mask"]["restrict_special_attention"])

In [30]:
cell_dataset = make_cell_dataset(
    dataset=train_dataset,
    vocab_size=vocab_size,
    seq_len_cell=params["data"]["seq_len_cell"],
    seq_len_neighborhood=params["data"]["seq_len_neighborhood"],
    max_special_tokens=max_special_tokens,
    tokenizer_type=params["data"]["tokenizer_type"],
    gt_type=params["meta"]["gt_type"],
    special_tokens=special_tokens,
    sampling_strategy=params["data"]["sampling_strategy"])

In [35]:
10 / sum([2, 5, 7])

0.7142857142857143

In [32]:
cell_dataset[0]

(tensor([  2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12, 541, 268, 425,
         307, 303,  25,  22, 357,  29,  52,  36,  33, 315,  48, 411, 112, 429,
         272, 267,  92, 244,  34,  45,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0, 334,  31, 225,  33, 291,  22, 205,  67,
         169, 289, 317,  92,  34,  14,  65,  29,  50,  76, 336, 320,  71, 119,
         209, 363, 206, 375, 311, 419, 288,  39, 268, 211, 370, 213, 180, 309,
         125, 112, 164, 362, 191, 190, 259, 313, 137, 307, 368,  16,  98,  74,
         123, 350, 113, 296, 115, 182,  70, 310,  89,  36, 224,   8,  68,  78,
         268,  34, 425,  33, 238, 168, 429, 222, 370, 269, 288,  86, 373, 148,
          92,  24, 137, 225, 103,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0

In [ ]:
loader, sampler = init_dataloader_and_sampler(
    cell_dataset=train_cell_dataset,
    batch_size=batch_size,
    distributed=True,
    world_size=world_size,
    rank=rank,
    collate_fn=mask_collator,
    pin_memory=pin_memory,
    num_workers=num_workers,
    drop_last=False,
    persistent_workers=False)

In [29]:
_, train_loader, train__sampler = make_cell_dataset(
            dataset=train_dataset,
            vocab_size=vocab_size,
            collator=mask_collator,
            pin_mem=params["data"]["pin_mem"],
            num_workers=params["data"]["num_workers"],
            world_size=world_size,
            rank=rank,
            drop_last=False,
            seq_len_cell=params["data"]["seq_len_cell"],
            seq_len_neighborhood=params["data"]["seq_len_neighborhood"],
            has_cls=params["data"]["has_cls"])

KeyError: 'vocab_size'

In [ ]:
for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(train_loader):
    print(udata)
    print(masks_enc)
    print(masks_pred)
    print(masks_attention)
    break